# Public OrcaPen Xmon — Q3D Electrostatic Boundary Alignment

Two fresh capacitance-only Q3D runs compare the same physical coupon
inside an open/infinite Region and a finite grounded Region enclosure.
The Xmon pad is the sole Signal; all physical grounds, couplers, and
bumps share one Ground net. The comparison records evidence only and
defines no solver-agreement gate.

This example uses SCGSim
[`v3.0.0rc6`](https://github.com/NCUPHY-LTLab/scgsim/tree/v3.0.0rc6)
(`4c3edca`), the revision this repository pins, installed with
`uv sync --locked --group aedt-notebooks`. SCGSim builds the 3D model
from the `GeometryPlan` snapshot, the same model Palace meshes, and Q3D
imports its STEP.

> **The earlier runs used another model route**
>
> The sealed 20260829 runs were prepared by SCGSim `1.0.0.dev15` from a
> GDS that this notebook drew layer by layer, and `v3.0.0rc6` does not
> read them. Their eight `QGB` objects were bound to the superconducting
> `Al` record instead of the PDK-authored `In` record; SCGSim lowered
> both records to native Q3D PEC. The runs analyzed by default here were
> prepared and run with `v3.0.0rc6`, which takes every material, the
> bumps’ `In` included, from the plan’s stack.

## Setup and Imports

In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import subprocess
from dataclasses import replace
from pathlib import Path

import gdsfactory as gf
from IPython.display import display
from scgsim.aedt import (
    MatrixRunControl,
    parse_aedt_spec,
    prepare_handoff,
    prepare_q3d_from_geometry,
    resolve_results,
)
from scgsim.geometry import GeometryPlan

import orcapen_sc_pdk
from orcapen_sc_pdk import LAYER_STACK, get_material_records
from orcapen_sc_pdk.helpers.assembly import place_flip_chip_ground_short_bumps

## Setup and Run Controls

In [ ]:
WORKFLOW_ACTION = "analyze_handoff"  # prepare_handoff | run | analyze_handoff
OUTPUT_ROOT = Path.cwd() / ".artifacts"
BOUNDARY_RUNS = (
    {
        "label": "open_region",
        "run_id": "kosen2024_xmon_q3d_l309p5_w24p65_g20_vac250_open_cg_rc6_20261122_01",
        "grounded_region_net": None,
    },
    {
        "label": "grounded_region",
        "run_id": "kosen2024_xmon_q3d_l309p5_w24p65_g20_vac250_grounded_cg_rc6_20261122_01",
        "grounded_region_net": "region_enclosure",
    },
)
RUN_DIRS = {case["label"]: OUTPUT_ROOT / case["run_id"] for case in BOUNDARY_RUNS}
RETURNED_RUN_DIRS = dict(RUN_DIRS)

COMPONENT_NAME = "kosen2024_flip_chip_xmon_qubit"
COMPONENT_PARAMETERS = {
    "bump_ring_count_per_side": 0,
    "qubit_pad_length": 309.5,
    "qubit_pad_width": 24.65,
    "qubit_gap": 20.0,
}
COUPON_PADDING_UM = 75.0
GROUND_SHORT_CLEARANCE_UM = 30.0
INDIUM_PLACEMENT_MODE = "corner_anchors"
VACUUM_PADDING_UM = (250.0, 250.0, 1000.0)  # x, y and z, on both sides of each.
AEDT_VERSION = "2024.2"
MODELING = "solid"  # Every metal, bump and substrate is a solid of its stack thickness.
DESIGN_NAME = "Kosen2024XmonQ3d"
SETUP_NAME = "Setup1"
EXTRACTION_FREQUENCY_GHZ = 4.7
MAXIMUM_PASSES = 20
CONVERGENCE_PERCENT = 0.1

if WORKFLOW_ACTION not in {"prepare_handoff", "run", "analyze_handoff"}:
    raise ValueError("WORKFLOW_ACTION must be prepare_handoff, run, or analyze_handoff.")

## Create Simulation Component / Coupon

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    orcapen_sc_pdk.activate()
    gf.clear_cache()
    device = gf.get_component(COMPONENT_NAME, **COMPONENT_PARAMETERS)
    coupon = place_flip_chip_ground_short_bumps(
        device,
        coupon_padding_um=COUPON_PADDING_UM,
        clearance_um=GROUND_SHORT_CLEARANCE_UM,
        placement_mode=INDIUM_PLACEMENT_MODE,
    )
    component = coupon.component
    plan = GeometryPlan(
        component=component,
        root_id="chip",
        layer_stack=LAYER_STACK,
        material_records=get_material_records(),
        coupon_padding_um=coupon.stack_coupon_padding_um,
    )
    for name, reference in coupon.named_instances.items():
        plan.register_instance(f"chip/{name}", reference)
    plan.set_nets({
        "xmon_pad": ("chip/device/D1_XMON_PAD",),
        "ground": (
            "chip/D0_TOP_GROUND_PLANE",
            "chip/D1_BOTTOM_GROUND_PLANE",
            *(f"chip/device/D1_COUPLER_{index}" for index in range(1, 5)),
            *(
                f"chip/{name}/D0_D1_INDIUM_BUMP"
                for name in coupon.named_instances if name.startswith("short_")
            ),
        ),
    })
    # Q3D's Region is the plan's vacuum region.
    plan.set_vacuum_region(padding=VACUUM_PADDING_UM)
    snapshot = plan.prepare()
    stack = snapshot.stack
    display(component)

## Initialize AEDT Project / App

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display(
        {
            "aedt_version": AEDT_VERSION,
            "pyaedt_version": importlib.metadata.version("pyaedt"),
            "scgsim_version": importlib.metadata.version("scgsim"),
        }
    )

## Import GDS and Build the HFSS/Q3D/Q2D Model

The PDK remains authoritative: the plan evaluates its typed face-metal
construction, and SCGSim builds one model of the coupon from the
snapshot. Both Region-boundary cases below use that one model.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    conductor_records = {item["semantic_id"]: item for item in stack["layers"]}
    bump_material_ids = {
        conductor_records[f"chip/{name}/D0_D1_INDIUM_BUMP"]["material_id"]
        for name in coupon.named_instances if name.startswith("short_")
    }
    if len(bump_material_ids) != 1:
        raise ValueError("Coupon bump instances must share one PDK material.")

## Geometry Verification

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    display(plan.entities)

## Materials and Boundaries

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    # The stack carries every material into the model; nothing is rebound here.
    display(
        {
            semantic_id: record["material_id"]
            for semantic_id, record in conductor_records.items()
        }
    )

## Ports / Nets / Excitations

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    # Capacitance only: the pad is the one Signal, every grounded conductor one Ground.
    NET_TYPES = {"xmon_pad": "Signal", "ground": "Ground"}
    PHYSICAL_GROUND_NETS = ("ground",)

## Simulation Setup

In [ ]:
run_control = MatrixRunControl(
    setup_name=SETUP_NAME,
    frequency_ghz=EXTRACTION_FREQUENCY_GHZ,
    maximum_passes=MAXIMUM_PASSES,
    convergence_percent=CONVERGENCE_PERCENT,
)

## Simulation Configuration

`prepare_q3d_from_geometry` leaves the Region open. A grounded Region is
a field of the Q3D request, not of the plan, so the grounded case is the
open case’s request on the same model, with the Region’s six faces made
a new Ground Net, `region_enclosure`.

In [ ]:
HANDOFFS = {}
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    open_case, grounded_case = BOUNDARY_RUNS
    open_handoff = prepare_q3d_from_geometry(
        snapshot,
        aedt_version=AEDT_VERSION,
        modeling=MODELING,
        output_dir=RUN_DIRS[open_case["label"]],
        project_name=open_case["run_id"],
        design_name=DESIGN_NAME,
        net_types=NET_TYPES,
        physical_ground_nets=PHYSICAL_GROUND_NETS,
        run_control=run_control,
    )
    HANDOFFS[open_case["label"]] = open_handoff
    open_spec = parse_aedt_spec(
        json.loads(open_handoff.spec_path.read_text(encoding="utf-8")),
        base_dir=open_handoff.run_dir,
    )
    HANDOFFS[grounded_case["label"]] = prepare_handoff(
        spec=replace(
            open_spec,
            project_name=grounded_case["run_id"],
            grounded_region_net=grounded_case["grounded_region_net"],
        ),
        output_dir=RUN_DIRS[grounded_case["label"]],
    )

## Solve and Export

In [ ]:
RUN_RETURN_CODES = {}
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    for case in BOUNDARY_RUNS:
        label = case["label"]
        handoff = HANDOFFS[label]
        display(handoff)
        if WORKFLOW_ACTION == "run":
            completed = subprocess.run([str(handoff.script_path)], cwd=handoff.run_dir, check=False)
            RUN_RETURN_CODES[label] = completed.returncode

## Adaptive-Pass Convergence / Solver Diagnostics

In [ ]:
RESULTS = {}
if WORKFLOW_ACTION in {"run", "analyze_handoff"}:
    for case in BOUNDARY_RUNS:
        label = case["label"]
        result = resolve_results(RETURNED_RUN_DIRS[label])
        RESULTS[label] = result
        display(result.convergence)

## Results: Plots and Readable Tables

### Physics Analysis Results

In [ ]:
for label, result in RESULTS.items():
    display({"boundary_case": label, "rows": result.physics_results()})

### Simulation Performance / Benchmarks

In [ ]:
for label, result in RESULTS.items():
    display({"boundary_case": label, **result.simulation_benchmark()})

## Save and Release AEDT

In [ ]:
# The SCGSim runner saves the exact project and releases only its owned AEDT session.
for case in BOUNDARY_RUNS:
    label = case["label"]
    if label in RESULTS:
        display(RESULTS[label].project_path)
    elif label in HANDOFFS:
        display(HANDOFFS[label].archive_path)